# Agentic Analytics for Healthcare — Notebook with Prompts

**"The \$4M Bonus That Wasn't the Answer."** This notebook interleaves the **CRIT prompts** you
paste into GitHub Copilot Chat (the *markdown* cells) with **local pandas** versions of the analysis
the agent produces (the *code* cells). The code runs against the CSVs in [`./data`](./data) so you
can rehearse the whole story offline — in the live demo, the agent writes this against the Fabric
lakehouse and adds visualizations.

See [`README.md`](./README.md) for the full talk track and [`data-loading.md`](./data-loading.md)
for the Fabric load steps.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# find ./data whether run from healthcare/ or the repo root
here = Path.cwd()
DATA = next(p / "data" for p in [here, here / "healthcare", *here.parents]
            if (p / "data" / "epic.DepartmentDim.csv").exists())
print("Loading from:", DATA)

epic_dept   = pd.read_csv(DATA / "epic.DepartmentDim.csv")
epic_census = pd.read_csv(DATA / "epic.NursingUnitCensusFact.csv")
ps_job      = pd.read_csv(DATA / "peoplesoft.PS_JOB.csv")
ps_dept     = pd.read_csv(DATA / "peoplesoft.PS_DEPT_TBL.csv")
ps_reason   = pd.read_csv(DATA / "peoplesoft.PS_ACTION_REASON.csv")
kron_tc     = pd.read_csv(DATA / "kronos.TIMECARD_SUMMARY.csv")
kron_agy    = pd.read_csv(DATA / "kronos.AGENCY_HOURS.csv")
kron_pay    = pd.read_csv(DATA / "kronos.PAYCODE.csv")
mgr         = pd.read_excel(DATA / "unit_manager_scheduling_log.xlsx", sheet_name="Scheduling Log")

def cc4(s):
    "Every system encodes the unit's cost-center differently (8105 / CHR8105 / 08105-RN); take the 4-digit core."
    digits = re.sub(r"\D", "", str(s))
    return digits[-4:] if len(digits) >= 4 else np.nan

epic_dept["cc"] = epic_dept.EpicCostCenter.map(cc4)
NAME = epic_dept.set_index("cc").DepartmentName.to_dict()
NET  = epic_dept.set_index("cc").LegacyNetwork.to_dict()

print({n: len(d) for n, d in dict(epic_dept=epic_dept, epic_census=epic_census,
      ps_job=ps_job, kron_tc=kron_tc, kron_agy=kron_agy, mgr=mgr).items()})

Loading from: /mnt/c/dave/git/vibe-analytics/healthcare/data


{'epic_dept': 9, 'epic_census': 54, 'ps_job': 61, 'kron_tc': 353, 'kron_agy': 24, 'mgr': 20}


## Prompt 0 — Orient and load

```text
You are attached to a Microsoft Fabric Spark Notebook with an attached lakehouse. I have three
schemas: epic, peoplesoft, and kronos, plus a nurse manager's Excel (unit_manager_scheduling_log.xlsx).
Show me the tables and 5 sample rows of each; we'll run analytics together. Show all work.
```

## Step 1 — The obvious (wrong) answer
The turnover + agency dashboard, and the PeopleSoft termination reasons. This is what leadership
is staring at when they draft the \$4M bonus.

In [2]:
ps_job["cc"] = ps_job.DEPTID.map(cc4)

head  = ps_job.groupby("cc").size().rename("headcount")
terms = ps_job[ps_job.ACTION == "TER"].groupby("cc").size().rename("terms")
turn  = pd.concat([head, terms], axis=1)
turn["terms"] = turn["terms"].fillna(0).astype(int)
turn["turnover_pct"] = (turn.terms / turn.headcount * 100).round(0)
turn["unit"] = turn.index.map(NAME)
turn["network"] = turn.index.map(NET)

# naive agency roll-up by unit (FLOATPOOL rows have no cc and silently drop out)
kron_agy["cc"] = kron_agy.LABOR_ACCT.map(cc4)
agy_naive = (kron_agy.dropna(subset=["cc"])
             .assign(cost=lambda d: d.AGENCY_HRS * d.BILL_RATE)
             .groupby("cc").agg(agency_hrs=("AGENCY_HRS", "sum"),
                                agency_cost=("cost", "sum")).round(0))

dash = turn.join(agy_naive).fillna({"agency_hrs": 0, "agency_cost": 0})
dash = dash.sort_values("turnover_pct", ascending=False)
print("=== The dashboard leadership is looking at ===")
print(dash[["unit", "network", "headcount", "terms", "turnover_pct",
            "agency_hrs", "agency_cost"]].to_string())

print("\n=== PeopleSoft termination reasons (the apparent 'why') ===")
tr = ps_job[ps_job.ACTION == "TER"].merge(ps_reason, on=["ACTION", "ACTION_REASON"], how="left")
print(tr.DESCR.value_counts().to_string())
print("\nObvious read: Legacy Cherry is worst; people leave for Relocation/Personal -> 'pay them more'.")

=== The dashboard leadership is looking at ===
                   unit        network  headcount  terms  turnover_pct  agency_hrs  agency_cost
cc                                                                                             
8105    5 West Med/Surg  Legacy Cherry          7      4          57.0      2955.0     326110.0
8115   Progressive Care  Legacy Cherry          7      3          43.0      1372.0     149169.0
8101    4 East Med/Surg  Legacy Cherry          6      2          33.0      1492.0     163297.0
7130     Ortho Surgical  Legacy Valley          6      1          17.0         0.0          0.0
6101    5 East Med/Surg   Legacy Metro          7      1          14.0         0.0          0.0
7120       Surgical ICU  Legacy Valley          7      1          14.0         0.0          0.0
6120        Medical ICU   Legacy Metro          8      1          12.0         0.0          0.0
6110  7 Tower Telemetry   Legacy Metro          6      0           0.0         0.0       

## Step 2 — Think a meta-layer higher (don't accept it)

```text
> Context: Agency spend has exploded; the dashboard blames the Cherry campus and low pay
  (termination reasons are mostly Relocation/Personal). Leadership wants a $4,000,000 blanket RN
  bonus. VERIFY everything before relying on it — including whether the reason codes and the
  post-merger cost-center joins are trustworthy.
> Role: You are a nursing-workforce data scientist, deeply skeptical of hand-entered HR codes; you
  know SCHEDULE/FLOAT problems often masquerade as pay or "personal" problems. Start with EDA.
> Interview: I'm the CNO with the CFO. Do NOT tell me to approve the bonus. Give me FIVE competing
  hypotheses ranked by testability, with the exact table+query to CONFIRM or REFUTE each. Actively
  try to prove the "low pay" story WRONG.
> Task: Go!
```

## Step 3 — Kill the pay hypothesis

```text
> Task: Join PeopleSoft COMPRATE (+ Kronos OT) to RN turnover by unit. If pay drove turnover, the
  LOWEST-paid units should be worst. Tell me honestly whether the hypothesis survives.
```

In [3]:
tc = kron_tc.merge(ps_job[["EMPLID", "cc", "COMPRATE"]], on="EMPLID", how="left")

pay = tc.groupby("cc").agg(avg_base_rate=("COMPRATE", "mean")).round(1)
paycmp = (turn[["unit", "network", "turnover_pct"]].join(pay)
          .sort_values("turnover_pct", ascending=False))
print(paycmp.to_string())

r = paycmp.avg_base_rate.corr(paycmp.turnover_pct)
print(f"\nCorrelation(base pay, turnover) = {r:+.2f}")
print("If low pay caused turnover we'd expect a NEGATIVE correlation.")
print("It's POSITIVE: the highest-paid units churn worst -> the pay hypothesis is DEAD.")

                   unit        network  turnover_pct  avg_base_rate
cc                                                                 
8105    5 West Med/Surg  Legacy Cherry          57.0           56.9
8115   Progressive Care  Legacy Cherry          43.0           54.0
8101    4 East Med/Surg  Legacy Cherry          33.0           50.5
7130     Ortho Surgical  Legacy Valley          17.0           44.5
6101    5 East Med/Surg   Legacy Metro          14.0           50.9
7120       Surgical ICU  Legacy Valley          14.0           55.2
6120        Medical ICU   Legacy Metro          12.0           57.6
6110  7 Tower Telemetry   Legacy Metro           0.0           49.6
7101   3 North Med/Surg  Legacy Valley           0.0           44.6

Correlation(base pay, turnover) = +0.50
If low pay caused turnover we'd expect a NEGATIVE correlation.
It's POSITIVE: the highest-paid units churn worst -> the pay hypothesis is DEAD.


## Step 3b — Kill the "sicker patients / heavier workload" hypothesis

```text
> Task: Rule out 'the worst units just have sicker, heavier patients.' Use epic.NursingUnitCensusFact
  (AcuityIndex, RequiredNursingHours) to compare high-turnover units to stable ones.
```

In [4]:
epic_census["cc"] = epic_census.EpicCostCenter.map(cc4)
acu = epic_census.groupby("cc").agg(avg_acuity=("AcuityIndex", "mean")).round(2)
acmp = (turn[["unit", "turnover_pct"]].join(acu)
        .sort_values("turnover_pct", ascending=False))
print(acmp.to_string())
r = acmp.avg_acuity.corr(acmp.turnover_pct)
print(f"\nCorrelation(acuity, turnover) = {r:+.2f}")
print("The worst unit (5 West) runs AVERAGE acuity; the high-acuity Medical ICU is stable.")
print("Workload isn't the driver either.")

                   unit  turnover_pct  avg_acuity
cc                                               
8105    5 West Med/Surg          57.0        1.03
8115   Progressive Care          43.0        1.34
8101    4 East Med/Surg          33.0        1.05
7130     Ortho Surgical          17.0        1.10
6101    5 East Med/Surg          14.0        1.03
7120       Surgical ICU          14.0        1.80
6120        Medical ICU          12.0        1.87
6110  7 Tower Telemetry           0.0        1.16
7101   3 North Med/Surg           0.0        1.01

Correlation(acuity, turnover) = -0.17
The worst unit (5 West) runs AVERAGE acuity; the high-acuity Medical ICU is stable.
Workload isn't the driver either.


## Step 4 — Follow the symptoms in Kronos

```text
> Task: Plot OT %, float hours, and max consecutive shifts over time by unit. If specific units
  destabilize in a specific window, tell me which units, which months, and what they share.
```

In [5]:
tc["ot_pct"] = tc.OT_HRS / (tc.REG_HRS + tc.OT_HRS) * 100
sym = (tc.groupby(["cc", "MONTH"])
       .agg(ot_pct=("ot_pct", "mean"),
            float_hrs=("FLOAT_HRS", "sum"),
            consec=("MAX_CONSECUTIVE_SHIFTS", "mean")).round(1)
       .reset_index())
sym["unit"] = sym.cc.map(NAME)

print("Overtime % by unit and month:")
ot_piv = sym.pivot(index="unit", columns="MONTH", values="ot_pct").round(0)
print(ot_piv.to_string())

print("\nMax consecutive shifts by unit and month:")
cs_piv = sym.pivot(index="unit", columns="MONTH", values="consec").round(0)
print(cs_piv.to_string())
print("\nThree Legacy Cherry units (5 West, Progressive Care, 4 East) escalate sharply Apr-Jun.")
print("A pay problem doesn't switch on in April. What do these three share? -> the schedule.")

Overtime % by unit and month:
MONTH              2026-01  2026-02  2026-03  2026-04  2026-05  2026-06
unit                                                                   
3 North Med/Surg       7.0      7.0      7.0      6.0      7.0      7.0
4 East Med/Surg       16.0     16.0     15.0     20.0     20.0     20.0
5 East Med/Surg       10.0     10.0      9.0      9.0      9.0      9.0
5 West Med/Surg       21.0     21.0     22.0     30.0     31.0     31.0
7 Tower Telemetry      8.0      8.0      8.0      8.0      8.0      8.0
Medical ICU           10.0     10.0     10.0     10.0     10.0     11.0
Ortho Surgical         9.0      9.0      8.0      8.0      8.0      8.0
Progressive Care      19.0     18.0     18.0     25.0     23.0     24.0
Surgical ICU          10.0     11.0     11.0     12.0     11.0     11.0

Max consecutive shifts by unit and month:
MONTH              2026-01  2026-02  2026-03  2026-04  2026-05  2026-06
unit                                                           

## Step 5 — Data-quality reckoning #1: the reason codes

```text
> Task: The terminations cluster in that Apr-Jun window. What ACTION_REASON did PeopleSoft record?
  Is 'Scheduling/Work-Life' (SCH) ever used? Do you believe the reason codes?
```

In [6]:
term = ps_job[ps_job.ACTION == "TER"].copy()
term["term_mo"] = term.TERMINATION_DT.str[:7]
term["unit"] = term.cc.map(NAME)
view = (term[["unit", "term_mo", "ACTION_REASON"]]
        .merge(ps_reason[["ACTION_REASON", "DESCR"]], on="ACTION_REASON", how="left")
        .sort_values(["term_mo", "unit"]))
print(view.to_string(index=False))

n_sch = int((term.ACTION_REASON == "SCH").sum())
print(f"\n'SCH' (Scheduling/Work-Life) used {n_sch} time(s) across {len(term)} terminations,")
print("even though the schedule spike lines up exactly with these exits.")
print("DQ #1: the reason codes are MISCODED. 'Data quality is solid' was an assumption, not a fact.")

            unit term_mo ACTION_REASON                                      DESCR
 4 East Med/Surg 2026-04           CAR             Voluntary - Career Advancement
 5 West Med/Surg 2026-04           CAR             Voluntary - Career Advancement
 5 West Med/Surg 2026-04           REL                     Voluntary - Relocation
    Surgical ICU 2026-04           PER               Voluntary - Personal Reasons
 4 East Med/Surg 2026-05           REL                     Voluntary - Relocation
     Medical ICU 2026-05           PER               Voluntary - Personal Reasons
  Ortho Surgical 2026-05           REL                     Voluntary - Relocation
Progressive Care 2026-05           REL                     Voluntary - Relocation
Progressive Care 2026-05           CAR             Voluntary - Career Advancement
 5 East Med/Surg 2026-06           REL                     Voluntary - Relocation
 5 West Med/Surg 2026-06           SCH Voluntary - Scheduling / Work-Life Balance
 5 West Med/Surg

## Step 6 — Data-quality reckoning #2: the join

```text
> Task: Total agency hours by unit and reconcile to the units. The three legacy networks use
  different cost-center formats; watch for a shared/float-pool labor account that isn't a real unit.
  Does a naive join UNDER-count agency on the worst units, and by how much?
```

In [7]:
total_cost = (kron_agy.AGENCY_HRS * kron_agy.BILL_RATE).sum()
naive_cost = agy_naive.agency_cost.sum()
hidden = kron_agy[kron_agy.cc.isna()]          # the 0FLOATPOOL-CHR rows
hidden_cost = (hidden.AGENCY_HRS * hidden.BILL_RATE).sum()

print("Distinct labor accounts in kronos.AGENCY_HOURS:")
print(kron_agy.LABOR_ACCT.value_counts().to_string())
print(f"\nTotal agency spend (6 mo):        ${total_cost:>12,.0f}")
print(f"Captured by naive unit join:      ${naive_cost:>12,.0f}")
print(f"HIDDEN in 0FLOATPOOL-CHR:          ${hidden_cost:>12,.0f}  ({hidden.AGENCY_HRS.sum():.0f} hrs)")
print("\nDQ #2: post-merger, Cherry agency hours were booked to a float-pool account, so the")
print("dashboard UNDER-counted agency on exactly the units that were worst.")

Distinct labor accounts in kronos.AGENCY_HOURS:
LABOR_ACCT
08101-RN          6
0FLOATPOOL-CHR    6
08105-RN          6
08115-RN          6

Total agency spend (6 mo):        $     720,968
Captured by naive unit join:      $     638,576
HIDDEN in 0FLOATPOOL-CHR:          $      82,393  (761 hrs)

DQ #2: post-merger, Cherry agency hours were booked to a float-pool account, so the
dashboard UNDER-counted agency on exactly the units that were worst.


## Step 7 — The smoking gun: the nurse manager's Excel

```text
> Task: No system explains WHY April. Load unit_manager_scheduling_log.xlsx and bring in BedsClosed,
  SelfSchedulingEnabled, MandatoryOT, FloatPolicy, and the free-text Notes for those units/months.
```

In [8]:
cols = ["Month", "Unit", "BedsClosed", "SelfSchedulingEnabled", "MandatoryOT", "FloatPolicy"]
flags = mgr[(mgr.BedsClosed > 0) | (mgr.SelfSchedulingEnabled == "N") | (mgr.MandatoryOT == "Y")]
print(flags[cols].to_string(index=False))

print("\n--- The tribal knowledge no system captured (free-text notes) ---")
for _, r in mgr[mgr.Notes.notna() & (mgr.Notes.astype(str).str.strip() != "")].iterrows():
    print(f"[{r.Month}] {r.Unit}: {r.Notes}")

  Month             Unit  BedsClosed SelfSchedulingEnabled MandatoryOT       FloatPolicy
2026-04  4 East Med/Surg           0                     Y           Y   Volunteer-first
2026-05  4 East Med/Surg           0                     N           Y Punitive/assigned
2026-06  4 East Med/Surg           0                     N           Y Punitive/assigned
2026-04  5 West Med/Surg           6                     N           Y Punitive/assigned
2026-05  5 West Med/Surg           6                     N           Y Punitive/assigned
2026-06  5 West Med/Surg           6                     N           Y Punitive/assigned
2026-04 Progressive Care           0                     N           Y Punitive/assigned
2026-05 Progressive Care           4                     N           Y Punitive/assigned
2026-06 Progressive Care           4                     N           Y Punitive/assigned

--- The tribal knowledge no system captured (free-text notes) ---
[2026-02] Medical ICU: Stable. High acuity 

## Step 8 — Quantify and prescribe

```text
> Task: Quantify total agency spend (including the float-pool hours you recovered), show how
  concentrated it is on the three units, and contrast it with the $4,000,000 blanket bonus. Give me a
  prescriptive recommendation with an estimated dollar impact.
```

In [9]:
cherry_ccs   = [cc for cc, net in NET.items() if net == "Legacy Cherry"]
cherry_naive = agy_naive.loc[agy_naive.index.isin(cherry_ccs)].agency_cost.sum()
cherry_true  = cherry_naive + hidden_cost          # float-pool belongs to Cherry
annual       = total_cost * 2

avg_core_loaded = ps_job.COMPRATE.mean() * 1.4     # ~40% benefits/overhead load
avg_bill        = kron_agy.BILL_RATE.mean()
premium_per_hr  = avg_bill - avg_core_loaded
total_agy_hrs   = kron_agy.AGENCY_HRS.sum()
premium_6mo     = premium_per_hr * total_agy_hrs

print("================ DECISION MEMO ================")
print(f"Agency spend, 6 months:                 ${total_cost:>12,.0f}")
print(f"  concentrated on 3 Legacy Cherry units: ${cherry_true:>12,.0f}  ({cherry_true/total_cost:.0%})")
print(f"Annualized agency run-rate:             ${annual:>12,.0f}")
print(f"Agency premium over loaded core RN:      ${premium_per_hr:>5,.0f}/hr x {total_agy_hrs:,.0f} hrs"
      f" = ${premium_6mo:>10,.0f} / 6mo  (${premium_6mo*2:,.0f}/yr)")
print("----------------------------------------------")
print("Root cause: a SCHEDULE failure on 3 merged-campus units (closed beds, suspended")
print("self-scheduling, mandatory OT, punitive floating) -- NOT pay and NOT acuity.")
print()
print("Recommendation: do NOT spend $4,000,000 on a blanket bonus (it pays everyone and never")
print("touches the cause). Instead fix the schedule on 5 West, Progressive Care, and 4 East:")
print("  - restore self-scheduling            - cap consecutive shifts")
print("  - reform the float policy            - staff the closed-bed / renovation plan")
print(f"Target: recover the bulk of the ${annual:,.0f}/yr agency run-rate for a fraction of $4M.")

================ DECISION MEMO ================
Agency spend, 6 months:                 $     720,968
  concentrated on 3 Legacy Cherry units: $     720,969  (100%)
Annualized agency run-rate:             $   1,441,937
Agency premium over loaded core RN:      $   37/hr x 6,580 hrs = $   242,492 / 6mo  ($484,984/yr)
----------------------------------------------
Root cause: a SCHEDULE failure on 3 merged-campus units (closed beds, suspended
self-scheduling, mandatory OT, punitive floating) -- NOT pay and NOT acuity.

Recommendation: do NOT spend $4,000,000 on a blanket bonus (it pays everyone and never
touches the cause). Instead fix the schedule on 5 West, Progressive Care, and 4 East:
  - restore self-scheduling            - cap consecutive shifts
  - reform the float policy            - staff the closed-bed / renovation plan
Target: recover the bulk of the $1,441,937/yr agency run-rate for a fraction of $4M.


---
### Takeaways
- **Dashboards encode assumptions.** This one blamed pay + geography and *undercounted* the worst
  units because of a post-merger join defect.
- **"Good data quality" is a claim to test, not accept** — miscoded reason codes and cost-center
  drift both looked clean at the row level.
- **The deliverable is a decision and a dollar figure**, not a chart.